# Python, NumPy, pandas, Polars & DuckDB

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/01_python.ipynb)

Runnable examples from **Data/ML Handbook** — section `01_python`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/01_python/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s01_numpy.py`

In [ ]:
import numpy as np, time
x = np.random.default_rng(0).normal(size=1_000_000)
t = time.perf_counter(); s1 = sum(v * v for v in x); t_loop = time.perf_counter() - t
t = time.perf_counter(); s2 = x @ x; t_vec = time.perf_counter() - t
print(f"loop {t_loop*1e3:.0f} ms vs vectorised {t_vec*1e3:.2f} ms  -> {t_loop/t_vec:.0f}x faster, same result: {np.isclose(s1, s2)}")
A = np.arange(6).reshape(2, 3); b = np.array([10, 20, 30])
print("broadcast (2,3)+(3,) ->\n", A + b)
print("standardise columns:", ((A - A.mean(0)) / A.std(0)).round(2).tolist())

### `code/s01_pandas.py`

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(0)
df = pd.DataFrame({"user": rng.integers(1, 5, 12), "city": rng.choice(["Nairobi", "Mombasa"], 12),
                   "amount": rng.gamma(2, 50, 12).round(1),
                   "ts": pd.date_range("2026-01-01", periods=12, freq="5D")})
out = (df.assign(month=df.ts.dt.to_period("M"))
         .groupby(["city", "month"], observed=True)
         .agg(n=("amount", "size"), revenue=("amount", "sum"), p90=("amount", lambda s: s.quantile(.9)))
         .round(1))
print(out.head(4))
df["user_avg"] = df.groupby("user")["amount"].transform("mean")          # broadcast group stat to rows
df["rank_in_city"] = df.groupby("city")["amount"].rank(ascending=False)   # window-function equivalent
df["prev_amount"] = df.sort_values("ts").groupby("user")["amount"].shift(1)  # LAG
print(df[["user", "city", "amount", "user_avg", "rank_in_city", "prev_amount"]].head(4).round(1).to_string())

### `code/s01_polars_duckdb.py`

In [ ]:
import polars as pl, duckdb, numpy as np
n = 1_000_000
df = pl.DataFrame({"city": np.random.default_rng(0).choice(["A", "B", "C"], n),
                   "sales": np.random.default_rng(1).exponential(100, n)})
df.write_parquet("data/sales.parquet")
lazy = (pl.scan_parquet("data/sales.parquet")              # lazy: nothing read yet
          .filter(pl.col("sales") > 50)
          .group_by("city").agg(pl.len().alias("n"), pl.col("sales").mean().round(2).alias("avg"))
          .sort("city"))
print(lazy.collect())                                       # optimiser pushes the filter into the scan
print(duckdb.sql("SELECT city, count(*) n, round(avg(sales),2) avg FROM 'data/sales.parquet' "
                 "WHERE sales > 50 GROUP BY 1 ORDER BY 1").fetchall())